In [22]:
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import QAOAAnsatz

from qiskit_ibm_runtime import QiskitRuntimeService

from qopt_best_practices.sat_mapping import SATMapper

from qubo_qaoa.utils.swap_strategy import QUBOSwapStrategy
from qubo_qaoa.utils.lr_qaoa import get_LR_qaoa_circuit, get_hardware_LR_qaoa_circuit

from qiskit_qaoa.utils.hamiltonian_utils import get_Q_and_hamiltonian
from qiskit_qaoa.utils.circuit_graph_utils import circuit_to_graph, graph_to_operator

In [23]:
service = QiskitRuntimeService(name='us_instance')
backend = service.backend(name='ibm_boston')

In [24]:
filename = 'trivial'

data_file = f'/lustre/scratch127/qpg/jc59/new_qubo_formulation/oriented/qubo_data/qubo_data_{filename}.gfa.pkl'
Q, hamiltonian, offset, ising_offset = get_Q_and_hamiltonian(data_file)
num_qubits: int = hamiltonian.num_qubits

print('Compiling with line SWAP strategy')
swap_strat = QUBOSwapStrategy.from_line(range(num_qubits))
edge_colouring = {(i, i+1): i % 2 for i in range(num_qubits)}
edge_colouring.update({(i+1, i): i % 2 for i in range(num_qubits)})

qc = QAOAAnsatz(
    cost_operator=hamiltonian,
    reps = 1,
    flatten=True
)
graph = circuit_to_graph(qc, qc.parameters[1])



Compiling with line SWAP strategy


In [25]:
remapped_g, sat_map, min_sat_layers = SATMapper(timeout=60).remap_graph_with_sat(
    graph=graph, swap_strategy=swap_strat, max_layers = int(num_qubits)
)
if remapped_g is None or sat_map is None:
    raise Exception('Failed to find initial layout')

cost_op = graph_to_operator(remapped_g, swap_strat._num_vertices)


In [26]:
min_sat_layers

13

In [27]:
p = 1
delta_b = 0.63
delta_g = 0.16

In [59]:

phis = ParameterVector('ϕ', num_qubits)
fixed_hardware_qc, hardware_qc, abstract_circuit = get_hardware_LR_qaoa_circuit(
    p, delta_b, delta_g, num_qubits,
    cost_op, sat_map, backend, edge_colouring, swap_strat,
    None, phis=phis,
)

In [60]:
fixed_hardware_qc.parameters, hardware_qc.parameters

(ParameterView([ParameterVectorElement(ϕ[0]), ParameterVectorElement(ϕ[1]), ParameterVectorElement(ϕ[2]), ParameterVectorElement(ϕ[3]), ParameterVectorElement(ϕ[4]), ParameterVectorElement(ϕ[5]), ParameterVectorElement(ϕ[6]), ParameterVectorElement(ϕ[7]), ParameterVectorElement(ϕ[8]), ParameterVectorElement(ϕ[9]), ParameterVectorElement(ϕ[10]), ParameterVectorElement(ϕ[11]), ParameterVectorElement(ϕ[12]), ParameterVectorElement(ϕ[13]), ParameterVectorElement(ϕ[14]), ParameterVectorElement(ϕ[15]), ParameterVectorElement(ϕ[16]), ParameterVectorElement(ϕ[17])]),
 ParameterView([ParameterVectorElement(β[0]), ParameterVectorElement(γ[0]), ParameterVectorElement(ϕ[0]), ParameterVectorElement(ϕ[1]), ParameterVectorElement(ϕ[2]), ParameterVectorElement(ϕ[3]), ParameterVectorElement(ϕ[4]), ParameterVectorElement(ϕ[5]), ParameterVectorElement(ϕ[6]), ParameterVectorElement(ϕ[7]), ParameterVectorElement(ϕ[8]), ParameterVectorElement(ϕ[9]), ParameterVectorElement(ϕ[10]), ParameterVectorElement(ϕ[11

In [29]:
abstract_circuit.count_ops()

OrderedDict([('swap', 111),
             ('rzz', 89),
             ('ry', 54),
             ('rz', 36),
             ('measure', 18)])

In [61]:
default_fixed, default_abstract = get_LR_qaoa_circuit(
    p, delta_b, delta_g, num_qubits,
    hamiltonian, None, phis=phis, measure=True
)

15:08:27 - qubo_qaoa.utils.lr_qaoa - INFO - p = 1. Circuit depth: 35


In [62]:
default_fixed.count_ops()

OrderedDict([('rzz', 89),
             ('ry', 54),
             ('rz', 36),
             ('measure', 18),
             ('barrier', 1)])

In [63]:
abstract_circuit.depth(), default_abstract.depth()

(33, 35)

In [64]:
fixed_hardware_qc.count_ops()

OrderedDict([('sx', 951), ('rz', 640), ('cz', 469), ('measure', 18), ('x', 1)])

In [65]:
from qiskit import transpile
transpiled_default_abstract = transpile(default_abstract, backend)

In [66]:
transpiled_default_abstract.count_ops()

OrderedDict([('sx', 785),
             ('rz', 616),
             ('cz', 397),
             ('measure', 18),
             ('x', 3),
             ('barrier', 1)])

In [67]:
fixed_hardware_qc.depth(), transpiled_default_abstract.depth()

(441, 613)

In [69]:
from qiskit_qaoa.utils.circuit_graph_utils import circuit_to_graph, graph_to_operator, circuit_construction
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter

cost_op = graph_to_operator(remapped_g)
singles = cost_op[cost_op.paulis.z.sum(axis=-1) == 1]
doubles = cost_op[cost_op.paulis.z.sum(axis=-1) == 2]

init_state = QuantumCircuit(num_qubits)
for i in range(num_qubits):
    init_state.ry(phis[i], i)

mixer_layer_even = QuantumCircuit(num_qubits)
beta = Parameter("β")
if phis is not None:
    inv_sat_map = {v: k for k, v in sat_map.items()}
    for i in range(num_qubits): # Ignore mapping
        mixer_layer_even.ry(-phis[i], i)
        mixer_layer_even.rz(-2 * beta, i)
        mixer_layer_even.ry(phis[i], i)

circ_dict = circuit_construction(singles, doubles, backend, swap_strat, edge_colouring, {}, p, init_state=init_state, mixer_layer=mixer_layer_even)

backend_circ = circ_dict["backend"]

In [70]:
backend_circ.depth()

171

In [71]:
backend_circ.count_ops(), hardware_qc.count_ops(), transpiled_default_abstract.count_ops()

(OrderedDict([('rz', 829),
              ('sx', 790),
              ('cz', 377),
              ('delay', 238),
              ('measure', 18)]),
 OrderedDict([('sx', 951),
              ('rz', 640),
              ('cz', 469),
              ('measure', 18),
              ('x', 1)]),
 OrderedDict([('sx', 785),
              ('rz', 616),
              ('cz', 397),
              ('measure', 18),
              ('x', 3),
              ('barrier', 1)]))

In [47]:
backend_circ.draw(fold=-1)

global phase: π/4
            ┌──────────────┐┌─────────┐  ┌────┐      ┌───────┐      ┌──────────────────┐  ┌────┐ ┌───────┐                                        ┌────┐        ┌───────┐   ┌──────────┐   ┌────┐      ┌───────┐                                            ┌────┐                                   ┌───────────────┐                                                              ┌──────────────┐                            ┌────┐                            ┌────────┐      ┌────┐   ┌──────────┐                                         ┌────┐       ┌───────┐                          ┌────┐    ┌───────┐                                  ┌────┐    ┌───────┐                  ┌────┐  ┌───────┐                                                                                ┌────┐      ┌───────┐   ┌─────────────┐                                     ┌────┐                                                ┌────┐            ┌────────┐      ┌────┐   ┌──────────┐                    ┌────┐      ┌───────┐                                                                                                           ┌────┐  ┌───────┐  ┌─────────────┐                                        ┌────┐       ┌───────┐                                ┌────┐     ┌───────┐                      ┌────┐    ┌───────┐                                     ┌────┐       ┌───────┐                        ┌────┐      ┌───────┐                                                              ┌────┐    ┌───────┐  ┌─────────────────┐                          ┌────┐    ┌───────┐                                   ┌────┐     ┌───────┐          ┌────┐    ┌───────┐                              ┌────┐         ┌───────┐                   ┌────┐      ┌───────┐                                                                 ┌────┐    ┌───────┐   ┌──────────┐                               ┌────┐      ┌───────┐                                       ┌────┐         ┌───────┐                      ┌────┐   ┌───────┐                                    ┌────┐                          ┌──────────────┐                                          ┌────┐                ┌────────┐       ┌────┐        ┌──────────┐                                    ┌────┐  ┌───────┐                                                         ┌────┐     ┌───────┐                                  ┌────┐       ┌───────┐                          ┌────┐      ┌───────┐                                                                             ┌────┐    ┌───────┐  ┌──────────┐                   ┌────┐     ┌───────┐                                         ┌────┐         ┌───────┐           ┌───────────────┐                                                               ┌────┐     ┌────────────────┐     ┌────┐        ┌──────────┐                                                                        ┌─┐                        
 q_5 -> 47 ─┤ Delay(8[dt]) ├┤ Rz(π/2) ├──┤ √X ├──────┤ Rz(π) ├──────┤ Rz((-41.5)*γ[0]) ├──┤ √X ├─┤ Rz(π) ├─────────────────────────■──────────────┤ √X ├────────┤ Rz(π) ├───┤ Rz(γ[0]) ├───┤ √X ├──────┤ Rz(π) ├─────────────────────────────■──────────────┤ √X ├──────────────────────────■────────┤ Delay(16[dt]) ├─────────────────────────────────────────────────────■────────┤ Delay(8[dt]) ├─────────────────■──────────┤ √X ├───────────────────────■────┤ Rz(-π) ├──────┤ √X ├───┤ Rz(-π/2) ├───────────────────────────────■─────────┤ √X ├───────┤ Rz(π) ├────────────■─────────────┤ √X ├────┤ Rz(π) ├────────────────────■─────────────┤ √X ├────┤ Rz(π) ├───────■──────────┤ √X ├──┤ Rz(π) ├────────────────────────────────────────────────────────────────────────■───────┤ √X ├──────┤ Rz(π) ├───┤ Rz(10*γ[0]) ├─────────────────────────────■───────┤ √X ├───────────────────────────■────────────■───────┤ √X ├────■───────┤ Rz(-π) ├──────┤ √X ├───┤ Rz(-π/2) ├───────■────────────┤ √X ├──────┤ Rz(π) ├───────────────────────────────────────────────────────────────────────────────────────────────────■───────┤ √X ├──┤ Rz(π) ├──┤ Rz(11*γ[

In [50]:
hardware_qc.draw(fold=-1)

global phase: 0
           ┌─────────┐┌────┐  ┌───────┐   ┌──────────────────┐┌────┐┌───────┐           ┌────┐    ┌───────┐     ┌──────────┐   ┌────┐  ┌───────┐                      ┌────┐┌─────────┐                                                                                                                                                                                                 ┌────┐                                                                                                                      ┌────┐                    ┌────┐                                            ┌────┐                  ┌────┐                   ┌────┐                     ┌────┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      ┌────┐         ┌────┐                         ┌────┐                ┌────┐             ┌────┐                                                                                                                                                                                                                                                 ┌────┐           ┌────┐        ┌────┐              ┌────┐                                                         ┌────┐                       ┌────┐                                                                                                               ┌────┐                                                                                                      ┌────┐                ┌────┐        ┌────┐                         ┌────┐                           ┌────┐                              ┌────┐                             ┌────┐                     ┌────┐                  ┌─────────┐      ┌────┐     ┌───────┐                            ┌────┐    ┌──────────┐                   ┌────┐                            ┌────┐    ┌────────────┐                                                                                                                                                                                                                                                                                                                                        ┌────┐                                                                                                                    ┌────┐      ┌────┐           ┌─────────┐       ┌────┐      ┌───────┐                                                                                                             ┌────┐      ┌───────┐┌──────────┐┌────┐  ┌──────────┐                     ┌────┐   ┌─────────┐                                                                      ┌────┐                       ┌────┐                                    ┌─┐                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       